In [1]:
using Pkg

Pkg.activate("mnist")
#Pkg.add("MLDatasets")
#Pkg.add("Images")
Pkg.add("DataFrames")
using MLDatasets
using Images

  Activating project at `~/Desktop/DataAssim.jl/mnist`
┌ Warning: could not download https://pkg.julialang.org/registries
│   exception = Downloads.RequestError("https://pkg.julialang.org/registries", 6, "Could not resolve host: pkg.julialang.org", Downloads.Response(nothing, "https://pkg.julialang.org/registries", 0, "", Pair{String, String}[]))
└ @ Pkg.Registry /Users/mouhtal/.julia/juliaup/julia-1.11.5+0.aarch64.apple.darwin14/share/julia/stdlib/v1.11/Pkg/src/Registry/Registry.jl:77
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`


In [2]:
Pkg.develop(path="../Krylov.jl")   # change path to local Krylov fork
Pkg.develop(path="../JSOSolvers.jl") 

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`


In [3]:
F = MLDatasets.MNIST(split = :train)
A = F.features  # images d'entrée
b = F.targets   # labels
println("$(length(b)) images")

60000 images


In [4]:
using Random

# extraction des parties de A et b pertinentes
digits = (1, 7)
index_digits = findall(x -> x ∈ digits, b)

#n_samples = 6000  # taille  souhaitée
#idx_sub = index_digits[randperm(length(index_digits))[1:n_samples]]
b_digits = b[index_digits]
A_digits = A[:, :, index_digits]

# définition des deux classes
b_digits[b_digits .== digits[1]] .= 1
b_digits[b_digits .== digits[2]] .= -1

# reformulation de A sous forme d'une matrice
# chaque colonne du nouveau A_digits est la vectorisation d'une des images,
# i.e., l'empilement de ses colonnes les unes par-dessus les autres
A_digits = reshape(A_digits, size(A_digits, 1) * size(A_digits, 2), size(A_digits, 3))
A_digits = convert(Matrix{Float64}, A_digits) ./ 255

size(A_digits)

(784, 13007)

In [5]:
Pkg.add("ADNLPModels")

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/mnist/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/mnist/Manifest.toml`


In [6]:
using NLPModels, ADNLPModels
using LinearAlgebra
using SparseArrays

In [7]:

Ahat = Diagonal(b_digits) * sparse(A_digits)'

function f(x)
    r = Ahat * x               
    y = 100.0 .* (1 .- tanh.(r))
    return 0.5 * dot(y, y)
end
   


f (generic function with 1 method)

In [8]:
using JSOSolvers

In [9]:
using DataFrames
function run_solver(nlp, subsolver; memory=nothing)

    if memory === nothing
        stats = trunk(nlp,
            max_time=10000.0,
            max_iter=500,
            verbose=0,
            subsolver=subsolver
        )
    else
        stats = trunk(nlp,
            max_time=10000.0,
            max_iter=500,
            verbose=0,
            subsolver=subsolver,
            subsolver_kwargs=(memory=memory,)
        )
    end

    row = (
        solver = string(subsolver) * (memory === nothing ? "" : "_m$(memory)"),
        status = stats.status,
        norm_sol = norm(stats.solution),
        objective = stats.objective,
        iter = stats.iter,
        obj_eval = nlp.counters.neval_obj,
        grad_eval = nlp.counters.neval_grad,
        hprod = nlp.counters.neval_hprod,
        time = stats.elapsed_time
    )

    reset!(nlp)

    return row
end

run_solver (generic function with 1 method)

In [10]:
x0 = ones(size(A_digits, 1))
nlp = ADNLPModel(f, x0, backend = :optimized)

ADNLPModel - Model with automatic differentiation backend ADModelBackend{
  ReverseDiffADGradient,
  ReverseDiffADHvprod,
  EmptyADbackend,
  EmptyADbackend,
  EmptyADbackend,
  SparseReverseADHessian,
  EmptyADbackend,
}
  Problem name: Generic
   All variables: ████████████████████ 784    All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 784               free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: (  0.00% sparsity)   307720          linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                          

In [11]:
results = DataFrame()

push!(results, run_solver(nlp, :cg))
push!(results, run_solver(nlp, :lbfgs, memory=100))
push!(results, run_solver(nlp, :diom, memory=100))
push!(results, run_solver(nlp, :lbfgs, memory=50))
push!(results, run_solver(nlp, :diom, memory=50))

Row,solver,status,norm_sol,objective,iter,obj_eval,grad_eval,hprod,time
,String,Symbol,Float64,Float64,Int64,Int64,Int64,Int64,Float64
1,cg,first_order,9832.89,80000.4,25,48,26,368,10.9629
2,lbfgs_m100,first_order,9823.05,80000.5,25,48,26,357,10.4148
3,diom_m100,first_order,9824.23,80000.5,25,48,26,336,9.73856
4,lbfgs_m50,first_order,9823.05,80000.5,25,48,26,357,10.3149
5,diom_m50,first_order,9824.23,80000.5,25,48,26,336,9.972


In [12]:
using PrettyTables
pretty_table(results)

┌────────────┬─────────────┬──────────┬───────────┬───────┬──────────┬──────────
│     solver │      status │ norm_sol │ objective │  iter │ obj_eval │ grad_ev ⋯
│     String │      Symbol │  Float64 │   Float64 │ Int64 │    Int64 │     Int ⋯
├────────────┼─────────────┼──────────┼───────────┼───────┼──────────┼──────────
│         cg │ first_order │  9832.89 │   80000.4 │    25 │       48 │         ⋯
│ lbfgs_m100 │ first_order │  9823.05 │   80000.5 │    25 │       48 │         ⋯
│  diom_m100 │ first_order │  9824.23 │   80000.5 │    25 │       48 │         ⋯
│  lbfgs_m50 │ first_order │  9823.05 │   80000.5 │    25 │       48 │         ⋯
│   diom_m50 │ first_order │  9824.23 │   80000.5 │    25 │       48 │         ⋯
└────────────┴─────────────┴──────────┴───────────┴───────┴──────────┴──────────
                                                               3 columns omitted
